## Context

Context 의 뜻을 풀어보면 아래와 같습니다. <br>
Context: **"어떤 정보나 상황을 이해하기 위해 함께 고려해야 하는 주변 정보나 환경"** <br>
Context Engineering 은 올바른 정보와 도구, 맞는 포맷을 AI 에게 전달하기 위해 진행됩니다. <br>

<br>

Context 는 두가지 분류로 나눌 수 있습니다.
1. 변환 가능성
- 정적 Context : 유저의 메타데이터, DB 커넥션 정보등이 들어갑니다.
- 동적 Context : 대화 내역, 중간 집계, 도구 호출 맥락 등이 들어갑니다.

2. 생명주기
- Runtime Context : 한개의 대화내용에서 실행과정에서 돌아가는 맥락입니다.
- Cross-Conversation Context : 여러 대화 내용에서 지속되는 맥락입니다.

<br>

이 튜토리얼에서는 각 Context 유형을 직접 구현하고 과제를 통해 실습합니다.

| 섹션 | 개념 | 핵심 패턴 |
|------|------|----------|
| 2 | 정적 Context | `RunnableConfig` — 고정 정보 전달 |
| 3 | Model Context (프롬프트) | `@dynamic_prompt` — 동적 시스템 프롬프트 |
| 4 | Model Context (메시지) | `@wrap_model_call` — 메시지 주입 |
| 5 | Tool Context | `Command` — State 에 쓰기 |
| 6 | Cross-Conversation | `InMemoryStore` — 장기 기억 |
| 7 | 종합 과제 | 위 모든 개념 통합 |

### 1. 환경설정

In [1]:
!pip install -U langchain langchain-openai langgraph python-dotenv


  Attempting uninstall: python-dotenv

    Found existing installation: python-dotenv 1.2.1

    Uninstalling python-dotenv-1.2.1:

      Successfully uninstalled python-dotenv-1.2.1

   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
  Attempting uninstall: langgraph
   ---------------------------------------- 0/2 [python-dotenv]
    Found existing installation: langgraph 1.0.9
   ---------------------------------------- 0/2 [python-dotenv]
   -------------------- ------------------- 1/2 [langgraph]
    Uninstalling langgraph-1.0.9:
   -------------------- ------------------- 1/2 [langgraph]
   -------------------- ------------------- 1/2 [langgraph]
      Successfully uninstalled langgraph-1.0.9
   -------------------- ------------------- 1/2 [langgraph]
   -------------------- ------------------- 1/2 [langgraph]
   -------------------- ------------------- 1/2 [langgraph]
   -------------------- -----------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from datetime import datetime
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model("gpt-4o-mini")
print("모델 초기화 완료!")

모델 초기화 완료!


### 2. 정적 Context (Static Context)

**정적 Context** 는 실행 중 변하지 않는 고정 정보입니다.
- 예: 유저 ID, 역할(role), API 키, DB 커넥션 정보

LangChain 에서는 `RunnableConfig` 를 통해 이 정보를 에이전트와 도구에 전달합니다.<br>
도구 함수의 파라미터에 `config: RunnableConfig` 를 선언하면 LangChain 이 자동으로 주입해줍니다.

```python
invoke(input, config={"configurable": {"user_id": "u001", "user_role": "editor"}})
                                          ↓ 자동 주입
def my_tool(..., config: RunnableConfig):
    user_role = config["configurable"].get("user_role")
```

In [2]:
# 정적 Context 예제: 사용자 역할(role)에 따라 삭제 권한을 제어하는 도구

@tool
def get_current_user(config: RunnableConfig) -> str:
    """현재 로그인한 사용자 정보를 반환합니다."""
    user_id = config["configurable"].get("user_id", "anonymous")
    user_role = config["configurable"].get("user_role", "viewer")
    return f"사용자 ID: {user_id}, 역할: {user_role}"

@tool
def delete_record(record_id: str, config: RunnableConfig) -> str:
    """레코드를 삭제합니다. 관리자(admin)만 사용할 수 있습니다."""
    user_role = config["configurable"].get("user_role", "viewer")
    if user_role != "admin":
        return f"❌ 권한 없음: '{user_role}' 역할은 레코드를 삭제할 수 없습니다."
    return f"✅ 레코드 '{record_id}'가 삭제되었습니다."

@tool
def read_record(record_id: str) -> str:
    """레코드를 읽습니다. 모든 역할이 사용할 수 있습니다."""
    return f"📄 레코드 '{record_id}' 내용: '중요 비즈니스 데이터입니다.'"

doc_agent = create_agent(model, tools=[get_current_user, delete_record, read_record])

In [3]:
# editor 역할로 실행 — 삭제 권한이 없어야 합니다
config_editor = {"configurable": {"user_id": "u001", "user_role": "editor"}}

response = doc_agent.invoke(
    {"messages": [{"role": "user", "content": "내 정보를 알려줘. 그리고 레코드 'doc-42'를 삭제해줘."}]},
    config=config_editor,
)
print("[editor 로 실행]")
response["messages"][-1].pretty_print()

print("\n" + "="*60 + "\n")

# admin 역할로 실행 — 삭제 권한이 있어야 합니다
config_admin = {"configurable": {"user_id": "u999", "user_role": "admin"}}
response = doc_agent.invoke(
    {"messages": [{"role": "user", "content": "내 정보를 알려줘. 그리고 레코드 'doc-42'를 삭제해줘."}]},
    config=config_admin,
)
print("[admin 으로 실행]")
response["messages"][-1].pretty_print()

[editor 로 실행]
================================== Ai Message ==================================

현재 사용자 정보는 다음과 같습니다:
- **사용자 ID**: u001
- **역할**: editor

레코드 'doc-42'를 삭제하려고 했지만, 역할이 'editor'인 경우에는 해당 레코드를 삭제할 수 없습니다. 관리자(admin) 권한이 필요합니다.


[admin 으로 실행]
================================== Ai Message ==================================

당신의 정보는 다음과 같습니다:

- 사용자 ID: u999
- 역할: admin

레코드 'doc-42'가 성공적으로 삭제되었습니다.


#### 🎯 과제 1: API 키 인증 도구 만들기

`config["configurable"]["api_key"]` 를 읽어 `"valid-key-2025"` 와 일치하면 `"✅ API 인증 성공"`,  
아니면 `"❌ 유효하지 않은 API 키"` 를 반환하는 `check_api_auth` 도구를 완성하세요.  
그리고 유효한 키 / 유효하지 않은 키 두 경우 모두 실행해보세요.

```python
config_valid   = {"configurable": {"api_key": "valid-key-2025"}}
config_invalid = {"configurable": {"api_key": "wrong-key"}}
```

In [4]:
@tool
def check_api_auth(config: RunnableConfig) -> str:
    """API 키 인증 상태를 확인합니다."""
    api_key = config["configurable"].get("api_key", "")
    # TODO: 키가 "valid-key-2025" 인지 확인하고 적절한 메시지 반환
    
    if api_key == "valid-key-2025":
        return "API 인증 성공"
    
    return "유효하지 않은 API 키"

# TODO: auth_agent 를 create_agent 로 생성
auth_agent = create_agent(model, tools=[check_api_auth])

# TODO: config_valid 로 실행
config_valid = {"configurable": {"api_key": "valid-key-2025"}}

auth_agent.invoke(
    {"messages": [{"role": "user", "content": "내 API Key 사용 가능해?"}]},
    config=config_valid,
)


# TODO: config_invalid 로 실행
config_invalid = {"configurable": {"api_key": "wrong-key"}}

auth_agent.invoke(
    {"messages": [{"role": "user", "content": "내 API Key 사용 가능해?"}]},
    config=config_invalid,
)


{'messages': [HumanMessage(content='내 API Key 사용 가능해?', additional_kwargs={}, response_metadata={}, id='e1dd9713-2953-4382-b90a-1e4c807c4298'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 47, 'total_tokens': 58, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DFqHk6sxlCCpjMNJENj5v8eWjHG80', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbb43-4fdf-7cb1-9f7f-100c6d225cb2-0', tool_calls=[{'name': 'check_api_auth', 'args': {}, 'id': 'call_TbHsauHJNrE06NYNOZdDcEgo', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 11, 'total_tok

### 3. Model Context — 동적 시스템 프롬프트

**Model Context** 는 각 LLM 호출 시 전달되는 지침·메시지·도구·포맷을 동적으로 제어합니다.

`@dynamic_prompt` 미들웨어를 사용하면 매 LLM 호출 직전에 시스템 프롬프트를 계산할 수 있습니다.  
`request.messages` 로 현재까지의 대화 내역에 접근 가능합니다.

```
LLM 호출 직전  →  @dynamic_prompt 실행  →  프롬프트 재계산  →  LLM 호출
```

> **Transient 업데이트**: 이 변경은 해당 호출에만 적용되며 State 에 저장되지 않습니다.

In [5]:
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def adaptive_prompt(request) -> str:
    """대화 길이에 따라 시스템 프롬프트를 자동 조정합니다."""
    msg_count = len(request.messages)
    base = "당신은 친절하고 도움이 되는 AI 어시스턴트입니다."

    if msg_count > 10:
        return base + "\n⚠️ 대화가 매우 길어졌습니다. 답변을 1~2문장으로 아주 간결하게 유지하세요."
    elif msg_count > 5:
        return base + "\n답변은 핵심만 간략히 작성하세요."
    else:
        return base + "\n자세하고 친절하게 답변해주세요."

adaptive_agent = create_agent(model, tools=[], middleware=[adaptive_prompt])

In [6]:
# 첫 번째 메시지 (msg_count 낮음 → 자세한 답변 기대)
response = adaptive_agent.invoke(
    {"messages": [{"role": "user", "content": "파이썬 리스트 컴프리헨션이란 무엇인가요?"}]}
)
print("[짧은 대화 - 자세한 답변]")
response["messages"][-1].pretty_print()

[짧은 대화 - 자세한 답변]
================================== Ai Message ==================================

파이썬 리스트 컴프리헨션(List Comprehension)은 리스트를 간결하고 효율적으로 생성하는 방법입니다. 일반적인 리스트를 생성하는 방식에 비해 코드의 가독성을 높이고, 더 짧고 명확하게 리스트를 만들 수 있게 해줍니다. 

리스트 컴프리헨션은 기본적으로 하나 이상의 `for` 문과 선택적 `if` 문을 결합하여 새로운 리스트를 생성합니다. 기본 구문은 다음과 같습니다:

```python
new_list = [expression for item in iterable if condition]
```

여기서:
- `new_list`는 새로 생성될 리스트입니다.
- `expression`은 리스트의 각 요소를 변환하기 위한 식입니다.
- `item`은 반복 가능한 객체(iterable)를 순회하는 현재 항목을 나타냅니다.
- `iterable`은 리스트, 튜플, 문자열, 또는 다른 이터러블 객체입니다.
- `if condition`은 선택적으로 추가할 수 있는 조건문으로, 이 조건을 만족하는 항목만 포함됩니다.

### 예시

1. **기본 리스트 컴프리헨션:**

   1부터 10까지의 제곱을 포함하는 리스트를 생성하는 예시입니다.

   ```python
   squares = [x**2 for x in range(1, 11)]
   print(squares)  # [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
   ```

2. **조건문을 포함한 리스트 컴프리헨션:**

   1부터 10까지의 숫자 중 짝수의 제곱만 포함하는 리스트를 만드는 예시입니다.

   ```python
   even_squares = [x**2 for x in range(1, 11) if x % 2 == 0]
   print(even_squares)  # [4, 16, 36,

In [7]:
# 대화가 길어진 상황 시뮬레이션 (msg_count > 10 → 간결한 답변 기대)
long_history = [
    {"role": "user",      "content": "안녕!"},
    {"role": "assistant", "content": "안녕하세요!"},
    {"role": "user",      "content": "파이썬 잘 알아?"},
    {"role": "assistant", "content": "네, 잘 압니다."},
    {"role": "user",      "content": "JavaScript 는?"},
    {"role": "assistant", "content": "JavaScript 도 잘 압니다."},
    {"role": "user",      "content": "오늘 날씨는?"},
    {"role": "assistant", "content": "저는 날씨 정보가 없습니다."},
    {"role": "user",      "content": "그렇구나."},
    {"role": "assistant", "content": "네!"},
    {"role": "user",      "content": "파이썬 리스트 컴프리헨션이란 무엇인가요?"},
]
response = adaptive_agent.invoke({"messages": long_history})
print("[긴 대화 - 간결한 답변]")
response["messages"][-1].pretty_print()

[긴 대화 - 간결한 답변]
================================== Ai Message ==================================

파이썬 리스트 컴프리헨션은 리스트를 간결하게 생성하는 방법으로, 기존 리스트나 반복 가능한 객체를 기반으로 새로운 리스트를 만드는 문법입니다.


#### 🎯 과제 2: 시간대에 따라 인사말 변경하기

`datetime.now().hour` 를 사용해 시간대별로 다른 인사말을 프롬프트에 포함하는  
`time_based_prompt` 미들웨어를 완성하세요.

| 시간 | 인사말 |
|------|--------|
| 오전 6~12시 | "좋은 아침입니다!" |
| 오후 12~18시 | "좋은 오후입니다!" |
| 오후 18~24시 | "좋은 저녁입니다!" |
| 그 외 (새벽) | "늦은 시각이네요. 수고 많으십니다!" |

시스템 프롬프트: `"당신은 AI 입니다. 모든 답변을 '{인사말}'로 시작하세요."`

In [8]:
@dynamic_prompt
def time_based_prompt(request) -> str:
    hour = datetime.now().hour
    # TODO: hour 에 따라 greeting 결정
    if hour < 12:
        greeting = "좋은 아침입니다!"
    elif hour < 18:
        greeting = "좋은 오후입니다!"
    elif hour < 24:
        greeting = "좋은 저녁입니다!"
    else:
        greeting = "늦은 시각이네요. 수고 많으십니다!"
    return f"당신은 AI입니다. 모든 답변을 '{greeting}'로 시작하세요."

time_agent = create_agent(model, tools=[], middleware=[time_based_prompt])

# TODO: time_agent 로 "안녕하세요!" 라고 말을 걸어보세요
response = time_agent.invoke({"messages": [{"role": "user", "content": "안녕하세요!"}]})
response["messages"][-1].pretty_print()

================================== Ai Message ==================================

좋은 아침입니다! 어떻게 도와드릴까요?


### 4. Model Context — 메시지 주입

`@wrap_model_call` 미들웨어를 사용하면 LLM 호출 직전에 메시지 목록을 수정할 수 있습니다.  
시스템 정보, 검색 결과, 사용자 컨텍스트 등을 동적으로 주입하는 데 사용합니다.

```
LLM 호출 직전  →  @wrap_model_call  →  메시지 추가/수정  →  LLM 호출  →  응답 처리
```

핵심 패턴:
```python
@wrap_model_call
def my_middleware(request, handler):
    new_msg = {"role": "user", "content": "주입할 내용"}
    request = request.override(messages=list(request.messages) + [new_msg])
    return handler(request)  # 반드시 handler 를 호출해야 합니다
```

In [9]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def inject_datetime(request, handler):
    """매 LLM 호출 전에 현재 날짜/시간 정보를 메시지로 주입합니다."""
    now = datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분")
    context_msg = {
        "role": "user",
        "content": f"[시스템 정보] 현재 일시: {now}"
    }
    updated_messages = list(request.messages) + [context_msg]
    request = request.override(messages=updated_messages)
    return handler(request)

datetime_agent = create_agent(model, tools=[], middleware=[inject_datetime])

In [10]:
# 에이전트는 도구 없이도 현재 시각을 알 수 있습니다
response = datetime_agent.invoke(
    {"messages": [{"role": "user", "content": "지금 몇 시야? 오늘 날짜도 알려줘."}]}
)
response["messages"][-1].pretty_print()

================================== Ai Message ==================================

현재 날짜는 2026년 3월 5일이고, 시간은 08시 52분입니다. 다른 질문이 있으시면 말씀해 주세요!


#### 🎯 과제 3: 사용자 이름을 메시지에 주입하기

`wrap_model_call` 을 사용하여 `username` 변수를 읽고  
`"[시스템 정보] 현재 사용자: {username}"` 메시지를 주입하는 `inject_username` 미들웨어를 완성하세요.  
에이전트가 사용자 이름을 알고 이름으로 불러줄 수 있게 해보세요.

힌트: `username` 은 미들웨어 바깥에서 변수로 정의하거나 하드코딩해도 됩니다.

In [15]:
username = "대훈"  # 원하는 이름으로 변경하세요

@wrap_model_call
def inject_username(request, handler):
    """현재 사용자 이름을 메시지에 주입합니다."""
    # TODO: username 을 포함한 context_msg 생성
    context_msg = {
        "role": "user",
        "content": f"현재 사용자: {username}"
    }
    # TODO: request.override 로 메시지 업데이트 후 handler 반환
    updated_messages = list(request.messages) + [context_msg]
    request = request.override(messages=updated_messages)

    return handler(request)

# TODO: username_agent 생성 후 "나를 이름으로 불러줘." 라고 요청해보세요
username_agent = create_agent(model,tools=[],  middleware=[inject_username])
response = username_agent.invoke({"messages": [{"role": "user", "content": "나를 이름으로 불러줘."}]})
response["messages"][-1].pretty_print()

================================== Ai Message ==================================

대훈님, 안녕하세요! 어떻게 도와드릴까요?


### 5. Tool Context — State 에 쓰기

도구는 단순히 결과를 반환하는 것 외에 **에이전트의 State 를 업데이트** 할 수 있습니다.  
`Command` 객체를 반환하면 도구 실행 후 State 가 영구적으로 변경됩니다.

```
도구 실행  →  Command(update={...}) 반환  →  State 업데이트  →  이후 턴에서 상태 참조
```

> **Persistent 업데이트**: Model Context 와 달리 State 변경은 대화 내내 유지됩니다.

In [ ]:
from typing import TypedDict
from langgraph.types import Command
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langchain.agents import AgentState

# 커스텀 State 정의 — 기본 messages 외에 추가 필드를 가집니다
class SessionState(AgentState):
    authenticated: bool
    access_level: str

@tool
def authenticate(password: str, runtime: ToolRuntime) -> Command:
    """비밀번호를 확인하고 인증 상태를 State 에 업데이트합니다."""
    if password == "langchain2025":
        return Command(update={
            "messages": [
                ToolMessage(
                    content="비밀번호 확인 성공",
                    tool_call_id=runtime.tool_call_id
                )
            ],
            "authenticated": True,
            "access_level": "full"
        })

    return Command(update={
        "messages": [
            ToolMessage(
                content="비밀번호가 틀렸습니다",
                tool_call_id=runtime.tool_call_id
            )
        ],
        "authenticated": False,
        "access_level": "none"
    })

@tool
def get_secret_report() -> str:
    """기밀 보고서를 반환합니다. (에이전트가 State 를 보고 접근 여부를 판단합니다)"""
    return "🔐 기밀 보고서: 2025년 매출 성장률 42%, 신규 사용자 +15만명"

# state_schema 로 커스텀 State 를 에이전트에 연결
auth_agent = create_agent(
    model,
    tools=[authenticate, get_secret_report],
    state_schema=SessionState,
)

In [46]:
# 잘못된 비밀번호로 시도
response = auth_agent.invoke({
    "messages": [{"role": "user", "content": "비밀번호 'wrong'으로 로그인하고 기밀 보고서 알려줘."}],
    "authenticated": False,
    "access_level": "none",
})
print("[인증 실패]")
response["messages"][-1].pretty_print()
print(f"→ State: authenticated={response.get('authenticated')}, level={response.get('access_level')}")

print("\n" + "="*60 + "\n")

# 올바른 비밀번호로 시도
response = auth_agent.invoke({
    "messages": [{"role": "user", "content": "비밀번호 'langchain2025'로 로그인하고 기밀 보고서 알려줘."}],
    "authenticated": False,
    "access_level": "none",
})
print("[인증 성공]")
response["messages"][-1].pretty_print()
print(f"→ State: authenticated={response.get('authenticated')}, level={response.get('access_level')}")

[인증 실패]
================================== Ai Message ==================================

비밀번호가 틀렸습니다. 올바른 비밀번호를 입력해 주시기 바랍니다.
→ State: authenticated=False, level=none


[인증 성공]
================================== Ai Message ==================================

비밀번호 확인 결과: **비밀번호 확인 성공**

기밀 보고서 내용:
- **모집 상태**: 2025년 목표 성과율 42%, 신규 사용자 +15만 명
→ State: authenticated=True, level=full


#### 🎯 과제 4: 방문 기록을 State 에 저장하기

`VisitState` 의 `last_page: str` 필드를 사용하여,  
`visit_page(page_name)` 도구가 `Command(update={"last_page": page_name})` 를 반환하도록 완성하세요.  
에이전트에게 `"홈 페이지에 방문했어. 마지막으로 방문한 페이지가 어디야?"` 라고 물어보세요.

힌트: 에이전트 시스템 프롬프트에 State 의 `last_page` 를 확인하라고 지시하면 더 잘 동작합니다.

In [ ]:
class VisitState(TypedDict):
    messages: list
    last_page: str

@tool
def visit_page(page_name: str) -> Command:
    """페이지를 방문하고 마지막 방문 페이지를 State 에 기록합니다."""
    # TODO: Command 를 반환해 last_page 를 page_name 으로 업데이트
    ...

# TODO: visit_agent 생성 (state_schema=VisitState)
visit_agent = ...

# TODO: 아래 메시지로 실행하고 State 에 last_page 가 저장되는지 확인하세요
# response = visit_agent.invoke({
#     "messages": [{"role": "user", "content": "홈 페이지에 방문했어. 마지막으로 방문한 페이지가 어디야?"}],
#     "last_page": "",
# })
# response["messages"][-1].pretty_print()
# print(f"→ State last_page: {response.get('last_page')}")

### 6. Cross-Conversation Context — 장기 기억 (Store)

**Cross-Conversation Context** 는 여러 대화 세션에 걸쳐 정보를 유지합니다.  
`InMemoryStore` 를 사용하면 대화가 끝나도 사용자 선호도, 기록 등을 보존할 수 있습니다.

```
1번 대화  →  Store 에 저장  →  2번 대화  →  Store 에서 불러오기
```

| | State | Store |
|---|---|---|
| 범위 | 단일 대화 | 여러 대화 |
| 유지 | 대화 종료 시 소멸 | 영속적 |
| 용도 | 인증 상태, 대화 흐름 | 사용자 설정, 히스토리 |

In [ ]:
from langgraph.store.memory import InMemoryStore

# Store 는 에이전트 외부에서 생성해 여러 대화 세션이 공유합니다
memory_store = InMemoryStore()

@tool
def save_preference(key: str, value: str, config: RunnableConfig) -> str:
    """사용자 선호도를 Store 에 저장합니다."""
    user_id = config["configurable"].get("user_id", "anonymous")
    existing = memory_store.get(("preferences",), user_id)
    prefs = existing.value if existing else {}
    prefs[key] = value
    memory_store.put(("preferences",), user_id, prefs)
    return f"✅ 저장 완료: {key} = '{value}'"

@tool
def get_preferences(config: RunnableConfig) -> str:
    """저장된 사용자 선호도를 Store 에서 불러옵니다."""
    user_id = config["configurable"].get("user_id", "anonymous")
    existing = memory_store.get(("preferences",), user_id)
    if not existing:
        return "저장된 선호도가 없습니다."
    return f"저장된 선호도: {existing.value}"

pref_agent = create_agent(model, tools=[save_preference, get_preferences])

In [ ]:
# --- 첫 번째 대화 세션 ---
print("=== 첫 번째 대화 세션 ===")
config_user = {"configurable": {"user_id": "u001"}}

response = pref_agent.invoke(
    {"messages": [{"role": "user", "content": "내 언어 선호도를 '한국어'로, 테마를 'dark'로 저장해줘."}]},
    config=config_user,
)
response["messages"][-1].pretty_print()

In [ ]:
# --- 두 번째 대화 세션 (새 invoke, 같은 Store) ---
print("=== 두 번째 대화 세션 (새 대화) ===")

response = pref_agent.invoke(
    {"messages": [{"role": "user", "content": "내 저장된 선호도를 알려줘."}]},
    config=config_user,
)
response["messages"][-1].pretty_print()

#### 🎯 과제 5: 최근 검색 키워드 저장하기

사용자가 검색할 때마다 `Store` 에 검색어를 리스트로 누적 저장하는  
`save_search` 도구를 완성하세요. (최대 5개, 오래된 것부터 제거)  
`get_search_history` 도구도 함께 만들어 최근 검색 기록을 보여주세요.

힌트: `existing.value` 는 이전에 저장한 파이썬 객체(리스트)가 그대로 돌아옵니다.

In [ ]:
search_store = InMemoryStore()

@tool
def save_search(keyword: str, config: RunnableConfig) -> str:
    """검색 키워드를 Store 에 저장합니다. (최근 5개 유지)"""
    user_id = config["configurable"].get("user_id", "anonymous")
    existing = search_store.get(("searches",), user_id)
    history = existing.value if existing else []
    # TODO: keyword 를 history 에 추가하고 최대 5개만 유지 (슬라이싱 활용)
    history = ...
    # TODO: search_store 에 저장
    ...
    return f"✅ 검색 저장: '{keyword}' (현재 {len(history)}개)"

@tool
def get_search_history(config: RunnableConfig) -> str:
    """최근 검색 기록을 반환합니다."""
    user_id = config["configurable"].get("user_id", "anonymous")
    # TODO: search_store 에서 기록 불러와 반환
    ...

# TODO: search_agent 생성
search_agent = ...

# TODO: 여러 번 검색 후 기록 조회해보세요
# 예시 흐름:
# 1. "LangChain 검색해줘"
# 2. "LangGraph 검색해줘"
# 3. "내 최근 검색 기록 알려줘"

### 7. 종합 과제 — Context Engineering 으로 고객 지원 에이전트 구축

지금까지 배운 모든 Context 개념을 통합하여 **고객 지원 에이전트** 를 구축합니다.

**요구사항:**

| 요구사항 | 사용 기술 |
|---------|----------|
| 사용자 등급(tier)에 따라 이용 가능한 기능 제한 | 정적 Context (`RunnableConfig`) |
| 대화가 길어지면 요약 모드로 전환 | Model Context (`@dynamic_prompt`) |
| 현재 날짜를 자동으로 컨텍스트에 포함 | Model Context (`@wrap_model_call`) |
| 티켓 생성 시 티켓 번호를 State 에 저장 | Tool Context (`Command`) |
| 사용자 문의 이력을 Store 에 누적 | Cross-Conversation (`InMemoryStore`) |

**도구 목록:**
- `create_ticket(issue)` — 지원 티켓 생성 → `Command` 로 State 업데이트
- `get_faq(question)` — FAQ 검색 (모든 등급)
- `get_premium_support(request)` — 전담 지원 (premium 전용, RunnableConfig 로 tier 확인)
- `save_inquiry(summary)` — 문의 이력 Store 에 저장
- `get_inquiry_history()` — 과거 문의 이력 조회

In [ ]:
from langchain.agents.middleware import dynamic_prompt, wrap_model_call
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command
from langchain_core.runnables import RunnableConfig
from typing import TypedDict

support_store = InMemoryStore()

# ─── State 정의 ────────────────────────────────────────────────
class SupportState(TypedDict):
    messages: list
    ticket_id: str
    ticket_open: bool

# ─── 도구 정의 (TODO 를 완성하세요) ─────────────────────────────

@tool
def create_ticket(issue: str, config: RunnableConfig) -> Command:
    """지원 티켓을 생성하고 State 를 업데이트합니다."""
    user_id = config["configurable"].get("user_id", "anonymous")
    ticket_num = f"TKT-{user_id[:3].upper()}-{datetime.now().strftime('%H%M%S')}"
    # TODO: Command 로 ticket_id=ticket_num, ticket_open=True 업데이트
    return Command(update={...})

@tool
def get_faq(question: str) -> str:
    """자주 묻는 질문을 검색합니다. (모든 등급 사용 가능)"""
    faqs = {
        "환불": "환불은 구매 후 7일 이내에 고객센터를 통해 신청 가능합니다.",
        "배송": "배송은 주문 후 3~5 영업일 내 완료됩니다.",
        "비밀번호": "로그인 화면에서 '비밀번호 찾기'를 클릭하세요.",
    }
    for key, answer in faqs.items():
        if key in question:
            return answer
    return "관련 FAQ 를 찾지 못했습니다. 티켓을 생성해 주세요."

@tool
def get_premium_support(request: str, config: RunnableConfig) -> str:
    """프리미엄 전담 지원을 제공합니다. (premium 등급 전용)"""
    # TODO: config 에서 tier 를 읽어 'premium' 이 아니면 거절 메시지 반환
    tier = ...
    if ...:
        return "❌ 이 기능은 premium 등급 사용자만 이용할 수 있습니다."
    return f"✅ 전담 매니저가 '{request}' 요청을 처리합니다. 30분 내 연락드립니다."

@tool
def save_inquiry(summary: str, config: RunnableConfig) -> str:
    """문의 내용을 Store 에 저장합니다."""
    user_id = config["configurable"].get("user_id", "anonymous")
    # TODO: support_store 에서 기존 이력 불러오고, 새 문의(summary + 현재 시각) 추가 후 저장
    ...
    return "✅ 문의 이력이 저장되었습니다."

@tool
def get_inquiry_history(config: RunnableConfig) -> str:
    """과거 문의 이력을 반환합니다."""
    user_id = config["configurable"].get("user_id", "anonymous")
    # TODO: support_store 에서 이력 불러와 반환
    ...

# ─── 미들웨어 (TODO 를 완성하세요) ──────────────────────────────

@dynamic_prompt
def support_prompt(request) -> str:
    # TODO: 메시지 수 > 8 이면 간결 모드, 그 외엔 상세 모드
    base = "당신은 친절한 고객 지원 AI입니다. 사용자의 문제를 적극적으로 해결하세요."
    ...

@wrap_model_call
def inject_date(request, handler):
    # TODO: 오늘 날짜(YYYY년 MM월 DD일)를 시스템 정보 메시지로 주입
    ...

# ─── 에이전트 생성 ──────────────────────────────────────────────
# TODO: support_agent 를 create_agent 로 생성
# tools=[create_ticket, get_faq, get_premium_support, save_inquiry, get_inquiry_history]
# middleware=[support_prompt, inject_date]
# state_schema=SupportState
support_agent = ...

# ─── 테스트 실행 ────────────────────────────────────────────────
config_basic   = {"configurable": {"user_id": "u001", "tier": "basic"}}
config_premium = {"configurable": {"user_id": "u002", "tier": "premium"}}

# TODO 1: basic 사용자가 premium 지원을 요청 → 거절 확인
# TODO 2: premium 사용자가 전담 지원 요청 → 승인 확인
# TODO 3: 티켓 생성 후 response.get('ticket_id') 로 State 저장 확인
# TODO 4: 두 번째 invoke 에서 문의 이력이 조회되는지 확인